In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/loans_sample.csv")
df.info()
print(df.isnull().sum())
df.describe().T

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 22 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   loan_amnt            250000 non-null  float64
 1   term                 250000 non-null  str    
 2   int_rate             250000 non-null  float64
 3   installment          250000 non-null  float64
 4   grade                250000 non-null  str    
 5   emp_length           235296 non-null  str    
 6   home_ownership       250000 non-null  str    
 7   annual_inc           250000 non-null  float64
 8   verification_status  250000 non-null  str    
 9   issue_d              250000 non-null  str    
 10  loan_status          250000 non-null  str    
 11  purpose              250000 non-null  str    
 12  addr_state           250000 non-null  str    
 13  dti                  249934 non-null  float64
 14  delinq_2yrs          250000 non-null  float64
 15  fico_range_low       250000 

,count,mean,std,min,25%,50%,75%,max
loan_amnt,250000.0,14433.642100,8725.454358,500.00,8000.0000,12000.00,20000.00,40000.00
int_rate,250000.0,13.237333,4.774441,5.31,9.7500,12.74,15.99,30.99
installment,250000.0,438.462906,261.486049,15.69,249.0775,374.79,581.58,1719.83
annual_inc,250000.0,76248.186431,69751.406002,0.00,45850.7500,65000.00,90000.00,9000000.00
dti,249934.0,18.263288,11.006084,0.00,11.8100,17.62,24.00,999.00
delinq_2yrs,250000.0,0.316732,0.877237,0.00,0.0000,0.00,0.00,26.00
fico_range_low,250000.0,696.177860,31.870049,660.00,670.0000,690.00,710.00,845.00
fico_range_high,250000.0,700.177964,31.870536,664.00,674.0000,694.00,714.00,850.00
open_acc,250000.0,11.584604,5.474487,0.00,8.0000,11.00,14.00,90.00
pub_rec,250000.0,0.214636,0.583398,0.00,0.0000,0.00,0.00,46.00


In [2]:
# Target: 1 = defaulted (Charged Off), 0 = repaid
df["default"] = (df["loan_status"] == "Charged Off").astype(int)

# " 36 months" -> 36
df["term"] = df["term"].str.extract(r"(\d+)").astype(int)

# "Dec-2015" -> a real date, then pull out the year
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["issue_year"] = df["issue_d"].dt.year

# "10+ years", "< 1 year", "3 years" -> numbers
def parse_emp(x):
    if pd.isna(x): return np.nan
    if x.startswith("<"): return 0
    if x.startswith("10+"): return 10
    return int(x.split()[0])

df["emp_length"] = df["emp_length"].apply(parse_emp)

# Merge rare categories
df["home_ownership"] = df["home_ownership"].replace({"ANY": "OTHER", "NONE": "OTHER"})
df[["term", "issue_year", "emp_length", "default"]].head()

,term,issue_year,emp_length,default
0,60,2014,6.0,1
1,36,2014,0.0,1
2,36,2017,9.0,0
3,36,2014,10.0,0
4,36,2012,10.0,0


In [3]:
print(df.isnull().mean().sort_values(ascending=False).round(4).head(8))

# Keep a flag so we remember which employment lengths were missing
df["emp_length_missing"] = df["emp_length"].isnull().astype(int)

# Numbers: fill with the median. Text columns: fill with "Unknown"
num_cols = df.select_dtypes(include="number").columns.drop("default")
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
cat_cols = df.select_dtypes(include="object").columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

print("Missing values left:", df.isnull().sum().sum())

emp_length     0.0588
revol_util     0.0006
dti            0.0003
loan_amnt      0.0000
installment    0.0000
grade          0.0000
int_rate       0.0000
term           0.0000
dtype: float64
Missing values left: 0


C:\Users\aanch\AppData\Local\Temp\ipykernel_18596\2226576222.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns


In [4]:
def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["loan_amnt", "annual_inc", "dti", "revol_bal", "revol_util"]:
    lo, hi = iqr_bounds(df[col])
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f"{col}: {n_out} outliers ({n_out/len(df):.2%}), bounds {lo:.1f} to {hi:.1f}")
    df[col] = df[col].clip(lower=max(lo, 0), upper=hi)   # cap, don't delete rows

loan_amnt: 1332 outliers (0.53%), bounds -10000.0 to 38000.0
annual_inc: 12170 outliers (4.87%), bounds -20373.1 to 156223.9
dti: 1011 outliers (0.40%), bounds -6.5 to 42.3
revol_bal: 14738 outliers (5.90%), bounds -14821.0 to 40555.0
revol_util: 8 outliers (0.00%), bounds -22.6 to 126.7


In [5]:
df["annual_inc"] = df["annual_inc"].clip(lower=df["annual_inc"].quantile(0.01))
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"]
df["income_bracket"] = pd.cut(
    df["annual_inc"],
    bins=[0, 30000, 60000, 90000, 120000, np.inf],
    labels=["<30k", "30-60k", "60-90k", "90-120k", "120k+"],
).astype(str)

df.to_csv("../data/processed/loans_clean.csv", index=False)
print(df.shape, "| missing:", df.isnull().sum().sum())
df.head()

(250000, 28) | missing: 0


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,...,pub_rec,revol_bal,revol_util,total_acc,default,issue_year,emp_length_missing,fico_avg,loan_to_income,income_bracket
0,10125.0,60,24.50,294.23,F,6.0,MORTGAGE,30000.0,Not Verified,2014-05-01,...,1.0,9463.0,47.6,26.0,1,2014,0,667.0,0.337500,<30k
1,3775.0,36,22.15,144.47,E,0.0,RENT,28000.0,Source Verified,2014-09-01,...,0.0,10461.0,45.9,17.0,1,2014,0,697.0,0.134821,<30k
2,9000.0,36,16.02,316.51,C,9.0,MORTGAGE,45000.0,Verified,2017-07-01,...,0.0,900.0,13.0,14.0,0,2017,0,712.0,0.200000,30-60k
3,7000.0,36,6.49,214.52,A,10.0,RENT,83000.0,Not Verified,2014-10-01,...,0.0,10415.0,16.5,47.0,0,2014,0,762.0,0.084337,60-90k
4,19125.0,36,8.90,607.29,A,10.0,MORTGAGE,73000.0,Not Verified,2012-12-01,...,0.0,12993.0,43.0,35.0,0,2012,0,722.0,0.261986,60-90k
